In [ ]:
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col,
    concat_ws,
    dayofweek,
    hour,
    lit,
    round,
    sha2,
    unix_timestamp,
    when,
)
from pyspark.sql.types import DoubleType, IntegerType, TimestampType

# Read Bronze.
bronze_df = spark.table("nyc_taxi.bronze.green_taxi")
quarantine_schema = "nyc_taxi.quarantine"
current_timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {quarantine_schema}")

# Cast and standardize source types.
cast_rules = {
    "lpep_pickup_datetime": TimestampType(),
    "lpep_dropoff_datetime": TimestampType(),
    "VendorID": IntegerType(),
    "RatecodeID": IntegerType(),
    "payment_type": IntegerType(),
    "passenger_count": IntegerType(),
    "fare_amount": DoubleType(),
    "extra": DoubleType(),
    "mta_tax": DoubleType(),
    "improvement_surcharge": DoubleType(),
    "tip_amount": DoubleType(),
    "tolls_amount": DoubleType(),
    "total_amount": DoubleType(),
    "trip_distance": DoubleType(),
    "pickup_longitude": DoubleType(),
    "pickup_latitude": DoubleType(),
    "dropoff_longitude": DoubleType(),
    "dropoff_latitude": DoubleType(),
}

df = bronze_df
for column_name, data_type in cast_rules.items():
    if column_name in df.columns:
        df = df.withColumn(column_name, col(column_name).cast(data_type))
    else:
        df = df.withColumn(column_name, lit(None).cast(data_type))

# Preserve provenance for values inferred during standardization.
silver_df = (
    df.withColumn(
        "passenger_count_was_imputed",
        col("passenger_count").isNull() | (col("passenger_count") <= 0) | (col("passenger_count") > 6),
    )
    .withColumn(
        "payment_type_was_imputed",
        col("payment_type").isNull() | (col("payment_type") < 1) | (col("payment_type") > 6),
    )
    .withColumn(
        "rate_code_was_imputed",
        col("RatecodeID").isNull() | (col("RatecodeID") < 1) | (col("RatecodeID") > 6),
    )
    .withColumn(
        "passenger_count",
        when(col("passenger_count").isNull() | (col("passenger_count") <= 0), lit(1))
        .when(col("passenger_count") > 6, lit(6))
        .otherwise(col("passenger_count")),
    )
    .withColumn(
        "payment_type",
        when(
            col("payment_type").isNull() | (col("payment_type") < 1) | (col("payment_type") > 6),
            lit(5),
        ).otherwise(col("payment_type")),
    )
    .withColumn(
        "RatecodeID",
        when(
            col("RatecodeID").isNull() | (col("RatecodeID") < 1) | (col("RatecodeID") > 6),
            lit(0),
        ).otherwise(col("RatecodeID")),
    )
)

# Add stable identifiers, refund provenance, and derived analytics fields.
silver_df = (
    silver_df.withColumn(
        "trip_id",
        sha2(
            concat_ws(
                "||",
                col("lpep_pickup_datetime"),
                col("lpep_dropoff_datetime"),
                col("PULocationID"),
                col("DOLocationID"),
            ),
            256,
        ),
    )
    .withColumn("is_reversed", (col("fare_amount") < 0) | (col("total_amount") < 0))
    .withColumn(
        "trip_duration_minutes",
        round(
            (unix_timestamp(col("lpep_dropoff_datetime")) - unix_timestamp(col("lpep_pickup_datetime"))) / 60,
            2,
        ),
    )
    .withColumn(
        "fare_per_mile",
        when(col("trip_distance") > 0, round(col("fare_amount") / col("trip_distance"), 2)),
    )
    .withColumn(
        "tip_percentage",
        when(col("fare_amount") > 0, round((col("tip_amount") / col("fare_amount")) * 100, 2)).otherwise(lit(0.0)),
    )
    .withColumn("pickup_hour", hour(col("lpep_pickup_datetime")))
    .withColumn("pickup_day_of_week", dayofweek(col("lpep_pickup_datetime")))
    .withColumn(
        "anomaly_flag",
        when(col("trip_distance") > 100, lit("EXTREME_DISTANCE"))
        .when(col("fare_amount") > 500, lit("EXTREME_FARE"))
        .otherwise(lit("NORMAL")),
    )
)

# Preserve negative fare/total records instead of silently losing refund evidence.
reversed_df = silver_df.filter(col("is_reversed"))
reversed_df.write.mode("overwrite").saveAsTable(f"{quarantine_schema}.taxi_reversed_{current_timestamp}")
reversed_df.write.mode("overwrite").saveAsTable(f"{quarantine_schema}.taxi_reversed_latest")

# Consume the duplicate quarantine produced by the upstream task.
duplicate_trip_ids = spark.table(f"{quarantine_schema}.taxi_trips_duplicates_latest").select("trip_id").distinct()

# Publish only valid analytical trips; rejected rows remain in quarantine.
silver_df = (
    silver_df.filter(col("lpep_pickup_datetime").isNotNull())
    .filter(col("lpep_dropoff_datetime").isNotNull())
    .filter(col("lpep_dropoff_datetime") > col("lpep_pickup_datetime"))
    .filter(col("trip_distance") > 0)
    .filter(~col("is_reversed"))
    .filter((col("trip_duration_minutes") > 0) & (col("trip_duration_minutes") <= 1440))
    .join(duplicate_trip_ids, "trip_id", "left_anti")
)

# Select the Silver contract.
silver_df = silver_df.select(
    "trip_id",
    col("VendorID").alias("vendor_id"),
    col("lpep_pickup_datetime").alias("pickup_datetime"),
    col("lpep_dropoff_datetime").alias("dropoff_datetime"),
    "trip_duration_minutes",
    "passenger_count",
    "passenger_count_was_imputed",
    "trip_distance",
    "fare_per_mile",
    "PULocationID",
    "DOLocationID",
    col("RatecodeID").alias("rate_code_id"),
    "rate_code_was_imputed",
    "store_and_fwd_flag",
    "payment_type",
    "payment_type_was_imputed",
    "tip_amount",
    "tip_percentage",
    "fare_amount",
    "extra",
    "mta_tax",
    "improvement_surcharge",
    "tolls_amount",
    "total_amount",
    "trip_type",
    "pickup_hour",
    "pickup_day_of_week",
    "is_reversed",
    "anomaly_flag",
)

table_name = "nyc_taxi.silver.green_taxi"
if spark.catalog.tableExists(table_name):
    existing_columns = spark.table(table_name).columns
    missing_columns = {
        "passenger_count_was_imputed": "BOOLEAN",
        "payment_type_was_imputed": "BOOLEAN",
        "rate_code_was_imputed": "BOOLEAN",
        "is_reversed": "BOOLEAN",
        "anomaly_flag": "STRING",
    }
    for column_name, data_type in missing_columns.items():
        if column_name not in existing_columns:
            spark.sql(f"ALTER TABLE {table_name} ADD COLUMNS ({column_name} {data_type})")

    DeltaTable.forName(spark, table_name).alias("target").merge(
        silver_df.alias("source"),
        "target.trip_id = source.trip_id",
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    silver_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
